# Paso 1: Definición del problema (Adaptado a Last.fm)

El problema de negocio:
Queremos saber qué factores hacen que una canción sea muy escuchada. ¿Es la duración? ¿Es el género musical (etiquetas)? ¿Es la popularidad general del artista?
Vamos a crear un modelo de Machine Learning que prediga si una canción será un "Hit" (alta popularidad) o "Estándar" (baja/media popularidad) basándonos en sus características.

¿Cómo lo convertimos en un problema de Machine Learning?
Será un problema de Clasificación.

A)_ Variable Objetivo (Target): En Last.fm, cada canción tiene un número de listeners (oyentes únicos) y playcount (reproducciones totales). Podemos definir que un "Hit" es una canción que supera cierto número de oyentes, clasificándolas en 1 (Hit) y 0 (No Hit).

B)_ Variables Predictoras (Features): Para cumplir el requisito de las 20 variables (incluyendo categóricas), podemos extraer:

_Nombre del artista (Categórica)

_Duración de la canción en milisegundos (Numérica)

_Oyentes totales del artista en la plataforma (Numérica)

_¿Tiene videoclip / está en Youtube? (Booleana/Categórica - algunos endpoints lo dan)

_al 20+. Las Etiquetas (Tags): Last.fm funciona con tags generados por usuarios ('rock', 'pop', 'female vocalists', '80s', 'acoustic', etc.). Tomaremos los 15-20 tags más populares de la plataforma y crearemos columnas binarias (1 si la canción tiene el tag 'rock', 0 si no).

# Paso 2: Obtención y carga del conjunto de datos

Para cumplir con la extracción de datos reales, usaremos la API pública de Last.fm.

In [2]:
import requests
import pandas as pd
import time
import os

API_KEY = 'c6975e1301d5981858f13766aa0b5774'
USER_AGENT = 'ProyectoBootcamp_Javier'
ARCHIVO_SALIDA = 'dataset_lastfm_60k_seguro.csv'

# Si existe un archivo de una prueba anterior, lo borramos para empezar limpios
if os.path.exists(ARCHIVO_SALIDA):
    os.remove(ARCHIVO_SALIDA)

print("Fase 1: Obteniendo lista de los 1.500 artistas más populares...")
lista_artistas = []

for pagina in range(1, 4):
    url_artistas = f"http://ws.audioscrobbler.com/2.0/?method=chart.gettopartists&api_key={API_KEY}&format=json&page={pagina}&limit=500"
    try:
        respuesta = requests.get(url_artistas, headers={'User-Agent': USER_AGENT})
        if respuesta.status_code == 200:
            datos = respuesta.json()
            artistas = datos['artists']['artist']
            for artista in artistas:
                lista_artistas.append(artista['name'])
    except Exception as e:
        print(f"Error al obtener artistas en página {pagina}: {e}")
    time.sleep(1)

print(f"¡Éxito! Se han encontrado {len(lista_artistas)} artistas distintos.")
print("\nFase 2: Extrayendo canciones con GUARDADO SEGURO...")

datos_temporales = [] # Usaremos esta lista como un "camión de carga" pequeño

# Recorremos la lista de artistas
for indice, artista in enumerate(lista_artistas):
    if indice % 50 == 0:
        print(f"Procesando artista {indice} de {len(lista_artistas)}: {artista}...")
        
    url_canciones = f"http://ws.audioscrobbler.com/2.0/?method=artist.gettoptracks&artist={artista}&api_key={API_KEY}&format=json&limit=50"
    
    try:
        respuesta = requests.get(url_canciones, headers={'User-Agent': USER_AGENT}, timeout=10)
        if respuesta.status_code == 200:
            datos = respuesta.json()
            canciones = datos.get('toptracks', {}).get('track', [])
            
            for cancion in canciones:
                datos_temporales.append({
                    'nombre_cancion': cancion.get('name'),
                    'nombre_artista': artista,
                    'oyentes': int(cancion.get('listeners') or 0),
                    'reproducciones': int(cancion.get('playcount') or 0),
                    'url': cancion.get('url')
                })
    except Exception as e:
        pass 
        
    time.sleep(1) # Pausa obligatoria
    
    # --- EL TRUCO SALVAVIDAS ---
    # Cada 50 artistas, o si es el último artista de la lista, volcamos los datos al CSV
    if (indice + 1) % 50 == 0 or (indice + 1) == len(lista_artistas):
        if datos_temporales:
            df_temp = pd.DataFrame(datos_temporales)
            
            # Si el archivo no existe, añade las cabeceras. Si ya existe, solo añade datos (mode='a')
            es_nuevo = not os.path.exists(ARCHIVO_SALIDA)
            df_temp.to_csv(ARCHIVO_SALIDA, mode='a', index=False, header=es_nuevo, encoding='utf-8')
            
            # Vaciamos el "camión" para liberar la memoria RAM
            datos_temporales = []

print(f"\n¡Extracción segura terminada! Todos tus datos están en '{ARCHIVO_SALIDA}'.")

# Leemos el archivo final para ver cuántas filas conseguimos realmente
df_final = pd.read_csv(ARCHIVO_SALIDA)
df_final = df_final.drop_duplicates(subset=['nombre_cancion', 'nombre_artista'])
df_final.to_csv(ARCHIVO_SALIDA, index=False) # Guardamos la versión sin duplicados

print(f"Total de filas únicas conseguidas: {len(df_final)}")

Fase 1: Obteniendo lista de los 1.500 artistas más populares...


KeyboardInterrupt: 